# ST-CDGM — Validation robuste et inférence stabilisée

Ce notebook charge un checkpoint sauvegardé et reconstruit les modèles pour :
- validation robuste sur les jeux de test par GCM,
- tests de non-régression,
- pipeline d'inférence stabilisée (vérifications de formes / NaN / Inf),
- export d'artefacts (CSV / JSON).

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Reproductibilite
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CONFIG = OmegaConf.load("config/training_config.yaml")
DEVICE = torch.device("cuda" if torch.cuda.is_available() and CONFIG.training.device == "cuda" else "cpu")

print(f"Device: {DEVICE}")
print("Config chargee depuis config/training_config.yaml")

In [ ]:
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import IntelligibleVariableConfig, IntelligibleVariableEncoder
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.evaluation.evaluation_xai import compute_temporal_variance_metrics

# Chemin explicite (optionnel) : None = recherche automatique dans save_dir
USER_CHECKPOINT_PATH = None  # ex: Path("models/st_cdgm_checkpoint_best.pth")
_env_ckpt = os.environ.get("ST_CDGM_CHECKPOINT", "").strip()
if _env_ckpt:
    USER_CHECKPOINT_PATH = Path(_env_ckpt)

_save_dir = Path(CONFIG.checkpoint.get("save_dir", "models"))
_candidates = []
if USER_CHECKPOINT_PATH:
    _candidates.append(Path(USER_CHECKPOINT_PATH))
for name in (
    "st_cdgm_checkpoint_best.pth",
    "st_cdgm_checkpoint.pth",
    "st_cdgm_checkpoint_last.pth",
):
    _candidates.append(_save_dir / name)

CHECKPOINT_PATH = next((p for p in _candidates if p.is_file()), None)
if CHECKPOINT_PATH is None and _save_dir.is_dir():
    _pth = sorted(_save_dir.glob("*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
    if _pth:
        CHECKPOINT_PATH = _pth[0]
        print(f"[INFO] Aucun nom standard — utilisation du .pth le plus recent: {CHECKPOINT_PATH}")

if CHECKPOINT_PATH is None or not CHECKPOINT_PATH.is_file():
    _msg = (
        "Aucun checkpoint PyTorch trouve. Chemins testes :\n  - "
        + "\n  - ".join(str(p) for p in _candidates)
        + (f"\n  - (aucun *.pth dans {_save_dir})" if _save_dir.is_dir() else f"\n  - (dossier absent: {_save_dir})")
        + "\n\nActions possibles :\n"
        "  1. Executer l'entrainement (st_cdgm_training_evaluation.ipynb ou train_ddp.py).\n"
        "  2. Definir la variable d'environnement ST_CDGM_CHECKPOINT=chemin/vers/fichier.pth\n"
        "  3. Dans cette cellule, definir USER_CHECKPOINT_PATH = Path(\"...\") (prioritaire sur la recherche auto)."
    )
    raise FileNotFoundError(_msg)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
required_keys = {"encoder_state_dict", "rcn_cell_state_dict", "diffusion_state_dict"}
missing_keys = required_keys.difference(checkpoint.keys())
if missing_keys:
    raise KeyError(f"Checkpoint invalide, cles manquantes: {missing_keys}")

print(f"Checkpoint charge: {CHECKPOINT_PATH}")
print(f"Epoch checkpoint: {checkpoint.get('epoch', 'n/a')}")

In [ ]:
# Configuration test (dossiers et GCM)
TEST_ROOT = Path("data/raw/test")
if not TEST_ROOT.exists():
    raise FileNotFoundError(f"Repertoire test introuvable: {TEST_ROOT}")

def test_lr_path(gcm):
    return str(TEST_ROOT / f"{gcm}_histupdated_compressed.nc")

def test_hr_path(gcm):
    return str(TEST_ROOT / f"{gcm}_historical_precip_compressed.nc")

def discover_test_gcms():
    lr_suffix = "_histupdated_compressed.nc"
    hr_suffix = "_historical_precip_compressed.nc"
    lr_stems = {f.name.replace(lr_suffix, "") for f in TEST_ROOT.glob(f"*{lr_suffix}")}
    hr_stems = {f.name.replace(hr_suffix, "") for f in TEST_ROOT.glob(f"*{hr_suffix}")}
    return sorted(lr_stems.intersection(hr_stems))

TEST_GCMS = discover_test_gcms()
if not TEST_GCMS:
    raise RuntimeError("Aucun couple LR/HR test detecte dans data/raw/test.")

print(f"GCMs detectes: {TEST_GCMS}")

In [ ]:
# Rebuild pipeline + builder + modeles depuis checkpoint
ckpt_cfg = checkpoint.get("config", {})
ckpt_cfg_full = checkpoint.get("config_full", {})

SEQ_LEN = int(ckpt_cfg.get("seq_len", CONFIG.data.seq_len))
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None

norm_cfg = ckpt_cfg.get("normalization", {})
MEAN_PATH = str(norm_cfg.get("mean_path", "data/raw/normalization_coefs/mean_1974_2011.nc"))
STD_PATH = str(norm_cfg.get("std_path", "data/raw/normalization_coefs/std_1974_2011.nc"))

LR_VARIABLES = list(ckpt_cfg.get("lr_variables", CONFIG.data.lr_variables))
STATIC_VARIABLES = list(ckpt_cfg.get("static_variables", CONFIG.data.static_variables))

# Convention test downscaling: cible toujours pr
TEST_HR_VARIABLES = ["pr"]

sampling_cfg = ckpt_cfg.get("sampling", {})
EVAL_NUM_STEPS = int(sampling_cfg.get("num_steps", CONFIG.diffusion.get("val_num_steps", 15)))
EVAL_SCHEDULER = sampling_cfg.get("scheduler_type", CONFIG.diffusion.get("scheduler_type", "ddpm"))

def build_test_pipeline(gcm):
    return NetCDFDataPipeline(
        lr_path=test_lr_path(gcm),
        hr_path=test_hr_path(gcm),
        static_path=STATIC_PATH,
        seq_len=SEQ_LEN,
        baseline_strategy=CONFIG.data.baseline_strategy,
        baseline_factor=CONFIG.data.get("baseline_factor", 4),
        normalize=bool(CONFIG.data.normalize),
        nan_fill_strategy=CONFIG.data.nan_fill_strategy,
        precipitation_delta=CONFIG.data.get("precipitation_delta", 0.01),
        lr_variables=LR_VARIABLES,
        hr_variables=TEST_HR_VARIABLES,
        static_variables=STATIC_VARIABLES,
        means_path=MEAN_PATH if Path(MEAN_PATH).exists() else None,
        stds_path=STD_PATH if Path(STD_PATH).exists() else None,
    )

probe_gcm = TEST_GCMS[0]
probe_pipeline = build_test_pipeline(probe_gcm)
probe_dataset = probe_pipeline.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)
probe_sample = next(iter(probe_dataset))

lr_shape = tuple(ckpt_cfg.get("lr_shape", CONFIG.graph.lr_shape))
hr_shape = tuple(ckpt_cfg.get("hr_shape", CONFIG.graph.hr_shape))
include_mid_layer = bool(ckpt_cfg_full.get("graph", {}).get("include_mid_layer", CONFIG.graph.include_mid_layer))

builder = HeteroGraphBuilder(
    lr_shape=lr_shape,
    hr_shape=hr_shape,
    static_dataset=probe_pipeline.get_static_dataset(),
    include_mid_layer=include_mid_layer,
)

allowed_nodes = set(builder.dynamic_node_types) | set(builder.static_node_types)
encoder_configs = [
    IntelligibleVariableConfig(
        name=mp.name,
        meta_path=(mp.src, mp.relation, mp.target),
        pool=mp.get("pool", "mean"),
    )
    for mp in CONFIG.encoder.metapaths
    if mp.src in allowed_nodes and mp.target in allowed_nodes
]
if probe_pipeline.get_static_dataset() is not None:
    encoder_configs.append(
        IntelligibleVariableConfig(
            name="static",
            meta_path=("SP_HR", "causes", "GP850"),
            pool="mean",
        )
    )

hr_channels = probe_sample["residual"].shape[1]
rcn_driver_dim = probe_sample["lr"].shape[1]

encoder = IntelligibleVariableEncoder(
    configs=encoder_configs,
    hidden_dim=CONFIG.encoder.hidden_dim,
    conditioning_dim=CONFIG.encoder.conditioning_dim,
).to(DEVICE)

rcn_cell = RCNCell(
    num_vars=len(encoder_configs),
    hidden_dim=CONFIG.rcn.hidden_dim,
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=CONFIG.rcn.dropout,
).to(DEVICE)

rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

diffusion = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=CONFIG.diffusion.height,
    width=CONFIG.diffusion.width,
    num_diffusion_steps=CONFIG.diffusion.steps,
    unet_kwargs=dict(
        layers_per_block=1,
        block_out_channels=(32,),
        down_block_types=("DownBlock2D",),
        up_block_types=("UpBlock2D",),
        norm_num_groups=8,
    ),
).to(DEVICE)

encoder.load_state_dict(checkpoint["encoder_state_dict"])
rcn_cell.load_state_dict(checkpoint["rcn_cell_state_dict"])
diffusion.load_state_dict(checkpoint["diffusion_state_dict"])

encoder.eval()
rcn_runner.cell.eval()
diffusion.eval()

print("Modeles reconstruits et poids charges depuis checkpoint.")
print(f"Sampling eval: steps={EVAL_NUM_STEPS}, scheduler={EVAL_SCHEDULER}")

In [ ]:
def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]

    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)

    dynamic_features = {node_type: lr_nodes_steps[0] for node_type in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)

    return {
        "lr": lr_tensor,
        "residual": sample["residual"],
        "baseline": sample.get("baseline"),
        "hetero": hetero,
    }


@torch.no_grad()
def generate_prediction_stable(sample):
    batch = convert_sample_to_batch(sample, builder, DEVICE)

    lr_data = batch["lr"].to(DEVICE)
    target = batch["residual"][-1].to(DEVICE)

    baseline_tensor = batch["baseline"][-1] if batch["baseline"] is not None else torch.zeros_like(target)
    baseline = baseline_tensor.to(DEVICE)

    H_init = encoder.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_output = rcn_runner.run(H_init, drivers, reconstruction_sources=None)

    conditioning = encoder.project_state_tensor(seq_output.states[-1]).to(DEVICE)
    generated = diffusion.sample(
        conditioning,
        num_steps=EVAL_NUM_STEPS,
        scheduler_type=EVAL_SCHEDULER,
        apply_constraints=False,
    )

    pred = generated.residual
    if pred.dim() == 4:
        pred = pred.squeeze(0)

    if pred.shape != target.shape:
        pred = F.interpolate(
            pred.unsqueeze(0),
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

    pred = torch.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)
    target = torch.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
    baseline = torch.nan_to_num(baseline, nan=0.0, posinf=0.0, neginf=0.0)
    return pred.cpu(), target.cpu(), baseline.cpu()


def compute_metrics_basic(full_pred, full_target):
    mask = ~torch.isnan(full_pred) & ~torch.isnan(full_target)
    if mask.sum() == 0:
        return {"MSE": float("nan"), "MAE": float("nan"), "Correlation": float("nan")}

    p = full_pred[mask].float()
    t = full_target[mask].float()

    mse = torch.mean((p - t) ** 2).item()
    mae = torch.mean(torch.abs(p - t)).item()

    p_c = p - p.mean()
    t_c = t - t.mean()
    eps = 1e-8
    denom = torch.sqrt((p_c ** 2).sum()) * torch.sqrt((t_c ** 2).sum()) + eps
    corr = ((p_c * t_c).sum() / denom).item() if denom.item() > eps else 0.0

    return {"MSE": mse, "MAE": mae, "Correlation": corr}

print("Inference stabilisee et metriques de base pretes.")

In [ ]:
# Evaluation robuste complete
N_TEST_SAMPLES = CONFIG.evaluation.get("max_test_samples", None) if CONFIG.get("evaluation") else None
if isinstance(N_TEST_SAMPLES, int) and N_TEST_SAMPLES <= 0:
    N_TEST_SAMPLES = None

rows = []
for gcm in TEST_GCMS:
    print(f"\n=== Evaluation GCM: {gcm} ===")
    pipe_test = build_test_pipeline(gcm)
    test_dataset = pipe_test.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)

    iterator = iter(test_dataset)
    n_done = 0
    while True:
        if N_TEST_SAMPLES is not None and n_done >= N_TEST_SAMPLES:
            break
        try:
            sample = next(iterator)
        except StopIteration:
            break

        pred, target, baseline = generate_prediction_stable(sample)
        full_pred = baseline + pred
        full_target = baseline + target

        full_pred = torch.nan_to_num(full_pred, nan=0.0, posinf=0.0, neginf=0.0)
        full_target = torch.nan_to_num(full_target, nan=0.0, posinf=0.0, neginf=0.0)

        m = compute_metrics_basic(full_pred, full_target)
        m.update(compute_temporal_variance_metrics(full_pred, full_target))
        m["gcm"] = gcm
        m["sample_idx"] = n_done
        rows.append(m)
        n_done += 1

    print(f"Echantillons evalues: {n_done}")

if not rows:
    raise RuntimeError("Aucune metrique calculee. Verifiez les donnees test/chemins/checkpoint.")

metrics_df = pd.DataFrame(rows)
gcm_summary_df = metrics_df.groupby("gcm", as_index=False).mean(numeric_only=True)
global_summary = metrics_df.mean(numeric_only=True).to_dict()

print("\nResume global:")
for k in ["MSE", "MAE", "Correlation", "temporal_var_rmse", "temporal_var_corr"]:
    print(f"  - {k}: {global_summary.get(k, float('nan')):.6f}")

gcm_summary_df

In [ ]:
# Non-regression + export artefacts
RESULTS_DIR = Path("results/validation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Seuils par defaut (ajustables)
DEFAULT_THRESHOLDS = {
    "MSE_MAX": float(CONFIG.evaluation.get("mse_max", 0.05)) if CONFIG.get("evaluation") else 0.05,
    "MAE_MAX": float(CONFIG.evaluation.get("mae_max", 0.2)) if CONFIG.get("evaluation") else 0.2,
    "CORR_MIN": float(CONFIG.evaluation.get("corr_min", 0.1)) if CONFIG.get("evaluation") else 0.1,
}

mse_ok = global_summary.get("MSE", float("inf")) <= DEFAULT_THRESHOLDS["MSE_MAX"]
mae_ok = global_summary.get("MAE", float("inf")) <= DEFAULT_THRESHOLDS["MAE_MAX"]
corr_ok = global_summary.get("Correlation", float("-inf")) >= DEFAULT_THRESHOLDS["CORR_MIN"]

non_reg_pass = bool(mse_ok and mae_ok and corr_ok)

report = {
    "checkpoint_path": str(CHECKPOINT_PATH),
    "n_samples": int(len(metrics_df)),
    "thresholds": DEFAULT_THRESHOLDS,
    "global_summary": {k: float(v) for k, v in global_summary.items()},
    "checks": {
        "mse_ok": mse_ok,
        "mae_ok": mae_ok,
        "corr_ok": corr_ok,
    },
    "non_regression_pass": non_reg_pass,
}

metrics_path = RESULTS_DIR / "validation_metrics_by_sample.csv"
gcm_path = RESULTS_DIR / "validation_metrics_by_gcm.csv"
report_path = RESULTS_DIR / "validation_report.json"

metrics_df.to_csv(metrics_path, index=False)
gcm_summary_df.to_csv(gcm_path, index=False)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"\nVerdict non-regression: {'PASS' if non_reg_pass else 'FAIL'}")
print(f"CSV sample-level: {metrics_path}")
print(f"CSV GCM-level: {gcm_path}")
print(f"JSON report: {report_path}")

report